# 01 — Event Loop with Coroutine Support

**Goal:** Extend the callback event loop to run `async def` coroutines. This is where `await` meets the event loop.

## What you need to build

### 1. `_Sleep` — an awaitable
A class with `__await__()` that yields `('sleep', seconds)` as a command to the event loop.

### 2. `Task` — wraps a coroutine
- `__init__(coro, loop)` — stores the coroutine and loop reference
- `step(value=None)` — calls `coro.send(value)`, gets back a command, handles it
- If `coro.send()` raises `StopIteration` → task is done
- If command is `('sleep', seconds)` → `loop.call_later(seconds, self.step)`
- Otherwise → `loop.call_soon(self.step)` (reschedule immediately)

### 3. EventLoop extensions
- `create_task(coro)` → creates a Task, schedules its first `step()`, returns the task
- `run(coro)` → creates main task, loops `_run_once()` until main task is done

### Test it:
```python
async def countdown(name, n):
    while n > 0:
        print(f"{name}: {n}")
        await sleep(1)
        n -= 1
    print(f"{name}: DONE!")

async def main():
    loop.create_task(countdown("Alpha", 3))
    loop.create_task(countdown("Beta", 5))
    await sleep(6)  # wait for all

loop.run(main())
```

Compare:
```python
# asyncio                          # Your loop
asyncio.run(main())                loop.run(main())
asyncio.create_task(coro)          loop.create_task(coro)
await asyncio.sleep(1)             await sleep(1)
```

Same pattern. You're building it.

In [ ]:
from collections import deque
import heapq
import time
import types

# Build: _Sleep, sleep(), Task, EventLoop with create_task() and run()
# You can copy your EventLoop from notebook 00 and extend it.


In [ ]:
# Test: concurrent async countdowns on YOUR event loop


### Question 1.1
You just ran `async def` functions with `await` on YOUR event loop — not asyncio's. How does the command flow work?

Trace: what happens when a coroutine does `await sleep(2)`?

*Your answer:*


## Challenge: Add `gather()` support

Implement `gather(*coros)` — waits for ALL coroutines to complete.

Hints:
1. `gather` creates Tasks for each coroutine
2. It suspends the caller until ALL tasks are done
3. One approach: return an awaitable that yields a `('wait_tasks', [tasks])` command
4. The event loop handles this by polling task completion

Test:
```python
async def main():
    await gather(
        countdown("A", 3),
        countdown("B", 2),
    )
    print("All done!")
```

In [ ]:
# Challenge — gather() support


## Summary

Your event loop now supports:
- `call_soon()` / `call_later()` — raw callbacks
- `create_task(coro)` — schedule a coroutine
- `run(coro)` — run until coroutine completes
- `await sleep(seconds)` — cooperative sleep

**Remaining notebooks (build when ready):**
- `02_add_io.ipynb` — Add `select()`-based I/O polling for async sockets
- `03_add_tasks.ipynb` — Full Task with result, exception, cancellation
- `04_full_loop.ipynb` — Complete event loop: tasks + I/O + timers + gather

**You built asyncio's core. The rest is just more features on the same foundation.**